In [36]:
%pip install pyomo gurobipy pandas

In [63]:
import gurobipy as gp
from gurobipy import GRB, Model

import pandas as pd
from typing import Any
import json


print("========== SOLVING INSTANCE OF PER ==========")

residents = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G",
]

# shift_types = [
#     "The one shift on this day",
#     # "3pm",
# ]

# dates = [
#     "2026-01-01",
#     "2026-01-02",
#     "2026-01-03",
# ]
dates = pd.date_range(start="2026-01-01", end="2026-02-01").strftime("%Y-%m-%d").tolist()
print(dates)

num_residents: int = len(residents)
# num_shift_types: int = len(shift_types)
num_dates: int = len(dates)


# init
m: Model = Model()
m.setParam('LogToConsole', 0)


x_rd = m.addVars(num_residents, num_dates, 
                    lb=0, ub=1, 
                    vtype=GRB.BINARY)

# set variable names
for (r, resident) in enumerate(residents):
    for (d, date) in enumerate(dates):
        x_rd[r, d].VarName = f"{resident}_{date}"

# one res for each date
for (d, date) in enumerate(dates):
    m.addConstr(gp.quicksum(x_rd[r, d] for (r, resident) in enumerate(residents)) == 1,
                name=f"single_shift_{date}")

# resident coverage ub of 7 across the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(gp.quicksum(x_rd[r, d] for (d, date) in enumerate(dates)) <= 7,
                name=f"coverage_ub_{resident}")
        
# obj fxn
m.setObjective(
    gp.quicksum(x_rd[r, d] for r, d in x_rd if r == "B"),
    GRB.MINIMIZE
)

# solve
m.optimize()

# report
scheduleReport: dict[str, Any] = {}
for (d, date) in enumerate(dates):
    workingResidentNames = [resident for (r, resident) in enumerate(residents) if x_rd[r, d].X == 1]

    daySchedule: dict[str, list[str]] = {}
    daySchedule["hardcoded single shift"] = workingResidentNames

    scheduleReport[date] = daySchedule
    print(f"{date}:\n")
    print(f"\t{daySchedule}\n")


# print vars
# for v in m.getVars():
#     print(f"{v.VarName} = {v.X}")

# infeasible: bool = m.Status == GRB.INFEASIBLE
# print(m.Status)



========== SOLVING INSTANCE OF PER ==========
['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04', '2026-01-05', '2026-01-06', '2026-01-07', '2026-01-08', '2026-01-09', '2026-01-10', '2026-01-11', '2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18', '2026-01-19', '2026-01-20', '2026-01-21', '2026-01-22', '2026-01-23', '2026-01-24', '2026-01-25', '2026-01-26', '2026-01-27', '2026-01-28', '2026-01-29', '2026-01-30', '2026-01-31', '2026-02-01']
Set parameter LogToConsole to value 0
2026-01-01:

	{'hardcoded single shift': ['A']}

2026-01-02:

	{'hardcoded single shift': ['C']}

2026-01-03:

	{'hardcoded single shift': ['C']}

2026-01-04:

	{'hardcoded single shift': ['E']}

2026-01-05:

	{'hardcoded single shift': ['D']}

2026-01-06:

	{'hardcoded single shift': ['F']}

2026-01-07:

	{'hardcoded single shift': ['E']}

2026-01-08:

	{'hardcoded single shift': ['G']}

2026-01-09:

	{'hardcoded single shift': ['A']}

2026-01-10:

	{'hardcoded 